# 몽글 멀티턴 플래너 — 코드 워크스루 (처음 보는 사람용)

이 노트북은 **`/v1/todo/chat`(멀티턴 플래너)** 가 사용자의 한 마디를 받아
TODO/일정 플랜으로 바꾸기까지, **파일별·코드별로** 어떻게 흘러가는지 설명한다.

> **읽는 법** — 위에서 아래로 순서대로. 각 절은 `① 무엇을` → `② 왜` → `③ 코드`
> → `④ 그림(mermaid)` 순서다. 그림은 GitHub·VS Code 에서 바로 렌더된다.

**대상 코드 상태:** 브랜치 `feat/planner-phase1-slot-schemas` (Phase 1 = 스키마 뱅크 +
plan_kind 분류 + routine allocator). **enrichment(D12 Tavily 시험일)은 "과하다"고 판단해
제거된 상태**이므로 이 문서에도 나오지 않는다.


## 실행 준비 (코드 셀을 돌리기 전에 — 가장 먼저 1번)

이 노트북은 `docs/features/todo/` 에 있어서, 코드 셀을 그냥 실행하면
`ModuleNotFoundError: No module named 'agents'` 가 난다(repo 루트가 import 경로에 없음).
**아래 셀을 가장 먼저 한 번 실행**하면 repo 루트를 `sys.path` 에 넣어 해결된다.
(또는 repo 루트에서 `uv run jupyter lab` 으로 노트북을 열어도 된다.)


In [3]:
# ── 실행 준비: repo 루트를 import 경로에 추가 (가장 먼저 1번 실행) ──
import sys
import pathlib

_root = pathlib.Path.cwd()
while not (_root / "agents").is_dir() and _root != _root.parent:
    _root = (
        _root.parent
    )  # CWD 에서 위로 올라가며 agents/ 가 z있는 폴더(=repo 루트)를 찾는다
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print("repo root =", _root, "| agents 발견:", (_root / "agents").is_dir())

repo root = /Users/jpaper/Documents/projects/mong-studio/mongle-ai | agents 발견: True


## 0. 한눈에 — planner 는 무엇을 하는 곳인가

사용자는 `"정처기 준비"`, `"매주 3번 헬스"`, `"운동 꾸준히 하고 싶어"` 처럼 **막연하거나
정보가 부족한** 목표를 던진다. planner 는:

1. 이 입력이 **계획으로 만들 수 있는 것인지** 판단하고(아니면 정중히 거절),
2. **정보가 부족하면 딱 필요한 것만 1개 되물어**(꼬리질문) 채우고,
3. 충분해지면 **날짜별 TODO/캘린더 후보**로 펼쳐 돌려준다.

핵심 설계 철학(설계서 D6·D8):

- **모델은 1개**, 그래프 골격은 **결정적(deterministic)**. 모델이 "툴을 쓸지" 정하지 않는다
  → 에이전트 궤적(tool-call trace) 학습을 피하고 SFT 분포를 1개로 유지.
- **작은 모델이 잘하는 것**(무엇을·분류·상대 순서)과 **코드가 잘하는 것**(날짜 산수·제약·검증)을
  나눈다(뉴로-심볼릭 분업).


## 1. 디렉토리 지도 — 어떤 파일이 무엇을 하나

```
agents/todo_creation/
├── schemas.py            # 입출력 타입: PlannerInput, GenerateResult/FollowUpResult/OutOfScopeResult, TaskCandidate
├── state.py              # ParsedGoal, PlanDay, Turn (그래프가 주고받는 데이터 모양)
├── protocols.py          # LLMPort (모델 어댑터가 지켜야 할 인터페이스)
├── config_utils.py       # get_ports(config) — 노드가 주입된 ports(llm) 를 꺼내는 헬퍼
└── planner/
    ├── pipeline.py       # ★ 진입점. run(): thread_id 로 신규/재개/수정/수락 분기 + 그래프 실행
    ├── graph.py          # ★ LangGraph 토폴로지 (노드·엣지 등록)
    ├── state.py          # PlannerGraphState (그래프 전용 state TypedDict)
    ├── goal_rules.py     # planner_node 가 쓰는 결정적 휴리스틱들 (마감 보정·위임·범위 등)
    ├── slot_schemas.py   # [Phase 1] 의도별 필수/선택 슬롯 뱅크 + missing_required()
    ├── allocator.py      # [Phase 1] routine cadence → 날짜 전개 (expand_routine)
    ├── date_parser.py    # "3일 뒤"·"다음주 화요일" → 절대 날짜
    └── nodes/
        ├── validate.py        # C2 입력 검증 (길이·한국어 비율) + history 누적
        ├── planner.py         # ★ 두뇌. judge_sufficiency 결과로 분기 결정
        ├── follow_up.py       # 꼬리질문 생성 + interrupt(일시정지)
        ├── plan_generator.py  # 플랜 생성 + 날짜 정리 + 마감 clamp
        └── out_of_scope.py    # 범위 밖 안내문

adapters/todo_creation/
├── qwen_llm.py           # ★ QwenLLM — LLMPort 구현 (judge_sufficiency / generate_plan / ...)
├── runpod_llm.py         # RunPodQwenLLM — QwenLLM 상속, complete_raw 만 RunPod 로
└── _prompts.py           # 모든 시스템/유저 프롬프트 문자열

api/todo_creation/router.py  # POST /v1/todo/chat → pipeline.run
```


## 2. 전체 흐름 한 장 (mermaid)

`★` 진입점부터 응답까지. 윗부분은 **`pipeline.run` 오케스트레이션**(어떤 state 로 그래프를
시작할지), 가운데 박스는 **LangGraph 그래프**다.

```mermaid
flowchart TD
    U(["사용자 메시지 (mode=multi)"]) --> RUN{"pipeline.run<br/>thread 상태?"}
    RUN -->|"신규 / thread 없음"| INIT["_initial_state"]
    RUN -->|"interrupt 에서 정지중"| RESUME["Command(resume=답변)"]
    RUN -->|"plan 존재 + 수락어(응/좋아)"| CACHE["직전 후보 그대로 반환"]
    RUN -->|"plan 존재 + 그 외"| REV["_revision_state (수정요청)"]

    INIT --> V
    REV --> V
    RESUME --> FU

    subgraph G["planner_graph (LangGraph · MemorySaver checkpoint)"]
        START((START)) --> V["validate"]
        V --> P["planner_node"]
        P -->|"sufficient"| PG["plan_generator"]
        P -->|"정보 부족 / 마감 애매"| FU["follow_up · interrupt"]
        P -->|"범위 밖"| OOS["out_of_scope"]
        FU -->|"resume 후 재평가"| P
        PG --> E((END))
        OOS --> E
    end

    FU -. "interrupt(질문)" .-> RFU(["FollowUpResult"])
    PG --> RGEN(["GenerateResult · todos+events+요약"])
    OOS --> ROOS(["OutOfScopeResult"])
    CACHE --> RGEN
    RFU -. "사용자가 답하면 다음 호출" .-> RUN
```

> 핵심: **follow_up 은 `interrupt()` 로 그래프를 멈추고** 질문만 돌려준다. 사용자가 답하면
> 다음 호출에서 `Command(resume=답변)` 으로 **멈춘 지점부터** 이어 달린다(thread_id 로 식별).


## 3. 진입점 — API → `pipeline.run`

### 3.1 라우터 (`api/todo_creation/router.py`)

```python
@router.post("/chat", response_model=Envelope[TurnResult])
async def chat(
    body: PlannerInput,
    ports: PlannerPorts = Depends(get_todo_planner_ports),  # llm 주입
) -> Envelope[TurnResult]:
    result = await multi_pipeline.run(body, ports=ports, now=datetime.now())
    return done(result)   # {"result": {...}} 봉투로 감쌈
```

입력 `PlannerInput`: `{mode:"multi", user_id, message(≤600자), today, thread_id?, user_profile_memory?}`.


### 3.2 `run()` 의 thread 분기 (`planner/pipeline.py`)

같은 대화(thread)를 여러 번 호출하므로, **지금이 첫 턴인지 / 질문에 답하는 중인지 / 이미 나온
플랜을 수정·수락하는 중인지** 를 먼저 가린다.

```python
async def run(input, *, ports, now) -> TurnResult:
    thread_id = input.thread_id or str(uuid4())
    config = {"configurable": {"ports": ports, "thread_id": thread_id}}

    if input.thread_id is not None:
        snapshot = _GRAPH.get_state(config)
        if snapshot.next:                              # ① follow_up interrupt 에서 멈춰있음
            graph_input = Command(resume=input.message)  #    → 답변으로 재개
        elif snapshot.values.get("plan"):              # ② 이미 플랜이 나온 상태
            if _is_acceptance(input.message):          #    "응/좋아/이대로" → 그대로 확정
                return _result_from_snapshot(...)
            graph_input = _revision_state(...)         #    그 외 → 수정요청으로 재실행
        else:
            graph_input = _initial_state(input, now)
    else:
        graph_input = _initial_state(input, now)       # ③ 신규 대화

    # 그래프를 stream 으로 돌리며 interrupt / 최종 state 수집 → TurnResult 로 변환
```

```mermaid
flowchart TD
    S(["run() 진입"]) --> T{"thread_id 있음?"}
    T -->|아니오| I["_initial_state · 신규"]
    T -->|예| SN["그래프 snapshot 조회"]
    SN --> N{"snapshot.next 있음?<br/>(interrupt 정지)"}
    N -->|예| R["Command(resume=메시지)"]
    N -->|아니오| PL{"snapshot 에 plan 있음?"}
    PL -->|"아니오"| I
    PL -->|"예 + 수락어"| C["직전 후보 그대로 반환"]
    PL -->|"예 + 그 외"| RV["_revision_state · 수정"]
```

> `_is_acceptance` 가 인정하는 수락어: `좋아/좋아요/응/네/예/그렇게 할게/이대로 할게/확정/...`
> (정규화 후 **정확히 일치**할 때만 — 오인식 방지).


In [5]:
# ── 실제 LangGraph 토폴로지를 코드가 직접 그려준다 (live mermaid) ──
# (먼저 맨 위 '실행 준비' 셀을 1번 실행해야 agents import 가 된다.
#  출력 mermaid 를 GitHub/VS Code 에 붙이면 렌더됨. draw_mermaid 는 langgraph 내장 — 추가 의존성 없음)
from agents.todo_creation.planner.graph import build_planner_graph

graph = build_planner_graph()
print(graph.get_graph().draw_mermaid())
open("planner_graph.png", "wb").write(graph.get_graph().draw_mermaid_png())


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	validate(validate)
	planner(planner)
	follow_up(follow_up)
	plan_generator(plan_generator)
	out_of_scope(out_of_scope)
	__end__([<p>__end__</p>]):::last
	__start__ --> validate;
	follow_up --> planner;
	out_of_scope --> __end__;
	plan_generator --> __end__;
	validate --> planner;
	planner -.-> plan_generator;
	planner -.-> follow_up;
	planner -.-> out_of_scope;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



20217

## 4. 상태(state) — 그래프가 주고받는 데이터

### 4.1 `PlannerGraphState` (`planner/state.py`)

노드들은 이 `TypedDict`(`total=False`)를 부분 갱신하며 전달한다. **각 노드는 dict 일부만
반환**하고 LangGraph 가 병합한다.

```python
class PlannerGraphState(TypedDict, total=False):
    # 입력 (pipeline 이 채움)
    message: str; today: date; now: datetime; user_id: str
    # 대화
    history: list[Turn]; recent_turns: list[Turn]
    user_profile_memory: dict | None; revision_request: str | None
    # planner 판단 결과
    sufficiency: bool | None; missing_aspects: list[str]
    parsed_goal: ParsedGoal | None
    follow_up_question: str | None; out_of_scope_message: str | None
    # 플랜 & 출력
    plan: list[PlanDay] | None; summary_text: str | None
    todos: list[TaskCandidate] | None; calendar_events: list[TaskCandidate] | None
```

### 4.2 `ParsedGoal` (`agents/todo_creation/state.py`) — planner 의 핵심 산출물

```python
class ParsedGoal(TypedDict, total=False):
    intent: Literal["plan", "out_of_scope"]
    plan_kind: Literal["exam", "routine", "vague_goal", "lifestyle"]   # [Phase 1]
    slots: dict[str, Any]                                              # [Phase 1] 채워진 슬롯
    goal_text: str; goal_tag: str
    deadline: date | None; daily_capacity_minutes: int | None
    revision_request: str | None; previous_plan: list[PlanDay]
    user_profile_memory: dict[str, Any]
```

> `plan_kind`/`slots` 가 Phase 1 에서 추가된 부분 — "이 목표가 어떤 종류인가 + 어떤 정보가
> 채워졌나" 를 담는다. 이게 **충족 판정과 꼬리질문의 근거**가 된다(5.3 참고).


## 5. 노드별 심층 분석

### 5.1 `validate` (`nodes/validate.py`) — 문지기

- **무엇을:** 메시지 길이(≤600), **한국어 음절 비율 ≥0.5** 검사. 통과하면 user turn 을
  `history` 에 append.
- **왜:** 한국어 플래너이므로 영어/잡음 입력을 초입에서 거른다(C2 규칙).

```python
if _korean_syllable_ratio(message) < 0.5:
    raise ValidationError(code="C2", message="...")
return {"history": history + [{"role": "user", "content": message}], ...}
```


### 5.2 `planner_node` (`nodes/planner.py`) — 두뇌

전체 분기를 결정하는 곳. `Command(goto=...)` 로 다음 노드를 직접 지정한다.

```python
async def planner_node(state, config) -> Command[str]:
    # ① 수정요청 + 기존 plan → 바로 재생성
    if state.get("revision_request") and state.get("plan"):
        return _plan_command(parsed_goal=<수정 goal>, sufficient=True, missing=[])

    # ② 꼬리질문 2회 캡(D9) → 기본값으로 진행
    if _follow_up_count(state.get("history", [])) >= 2:
        return _plan_command(parsed_goal=build_recovery_goal(state), sufficient=True, missing=[])

    # ③ 모델에게 충족 판정 위임
    sufficient, missing, parsed = await llm.judge_sufficiency(history=..., message=..., today=...)

    # ④ out_of_scope 처리 (단, 짧고 명백할 때만 수용 — 아니면 plan 으로 복구)
    if parsed.intent == "out_of_scope" and should_accept_out_of_scope(state): ... goto="out_of_scope"

    # ⑤ 사용자 입력의 명시 날짜를 deadline 으로 보정
    merge_deadline_from_state(state, resolved_goal)

    # ⑥ 시험/마감류인데 기한이 애매하면 → 마감일을 되묻는다
    if sufficient and needs_deadline_follow_up(state, resolved_goal):
        return Command(goto="follow_up", update={"missing_aspects": ["deadline"], ...})

    # ⑦ 최종: 충분하면 plan_generator, 아니면 follow_up
    return Command(goto="plan_generator" if sufficient else "follow_up", update={...})
```

```mermaid
flowchart TD
    A(["planner_node"]) --> C1{"수정요청 + 기존 plan?"}
    C1 -->|예| PGEN["→ plan_generator (수정)"]
    C1 -->|아니오| C2{"꼬리질문 2회 이상?"}
    C2 -->|"예(캡)"| RECOV["recovery goal → plan_generator"]
    C2 -->|아니오| J["judge_sufficiency (5.3)"]
    J --> C3{"intent=out_of_scope?"}
    C3 -->|예| OOS["→ out_of_scope"]
    C3 -->|아니오| MG["merge_deadline_from_state"]
    MG --> C4{"sufficient + 마감 애매한 시험/마감류?"}
    C4 -->|예| FU["→ follow_up: 마감 질문"]
    C4 -->|아니오| C5{"sufficient?"}
    C5 -->|예| PGEN
    C5 -->|아니오| FU
```

**`goal_rules.py` 의 조연들** (모두 결정적·LLM 무관):

| 함수                         | 역할                                                                     |
| ---------------------------- | ------------------------------------------------------------------------ |
| `merge_deadline_from_state`  | "3일 뒤" 등 사용자 명시 날짜를 `deadline` 으로 채움 (`date_parser` 사용) |
| `needs_deadline_follow_up`   | 시험·마감·여행 등 **날짜 민감 목표**인데 기한이 없으면 되묻게 함         |
| `delegates_planning`         | "알아서 짜줘" 류 → 더 안 묻고 진행                                       |
| `should_accept_out_of_scope` | 아주 짧고 명백한 첫 입력만 범위 밖으로 인정(오거절 방지)                 |
| `build_recovery_goal`        | 캡/오판 시 기존 정보로 최소 목표 복구                                    |


### 5.3 `judge_sufficiency` (`adapters/.../qwen_llm.py`) — 분류 + 충족 판정 [Phase 1 핵심]

모델은 JSON 하나를 뱉고(`plan_kind`, `slots`, `goal_text`, `deadline`...), 코드가 **충족 여부**를
다음과 같이 가른다:

```python
plan_kind = goal.get("plan_kind")
slots = goal.get("slots") if isinstance(goal.get("slots"), dict) else {}
if isinstance(plan_kind, str) and plan_kind in SLOT_SCHEMAS:
    goal["plan_kind"] = plan_kind
else:
    plan_kind = None                      # 비정상/미분류 → 폴백

# 일상 3종은 스키마 뱅크로 '코드가' 충족을 결정 (모델 자평 무시)
if intent == "plan" and plan_kind in {"routine", "vague_goal", "lifestyle"}:
    filled = {k for k, v in slots.items() if v not in (None, "", [], {})}
    missing = missing_required(plan_kind, filled)
    return (not missing), missing, goal

# exam / 미분류 → 모델의 is_sufficient·missing_aspects 그대로 (기존 거동 보존)
return bool(parsed["is_sufficient"]), missing_aspects, goal
```

```mermaid
flowchart TD
    J(["judge_sufficiency"]) --> M["LLM → JSON(plan_kind, slots, ...)"]
    M --> G{"plan_kind 유효한 str?"}
    G -->|아니오| F["plan_kind=None (폴백)"]
    G -->|예| K{"종류?"}
    F --> MODEL
    K -->|"routine/vague_goal/lifestyle"| SB["missing_required(스키마 뱅크)<br/>코드가 충족 결정"]
    K -->|"exam / 미분류"| MODEL["모델 is_sufficient·missing 그대로"]
    SB --> O(["(sufficient, missing, parsed_goal)"])
    MODEL --> O
```

> **왜 이렇게?** 시험(exam)은 이미 기존 휴리스틱(마감일 등)이 잘 동작하므로 건드리지 않고,
> 새로 들어온 일상 3종만 **선언적 스키마**로 "무엇이 더 필요한지" 를 코드가 결정한다.


#### 슬롯 뱅크 (`planner/slot_schemas.py`)

```python
SLOT_SCHEMAS = {
  "exam":      required=(exam_part, exam_date, daily_hours, current_level), optional=(...)
  "routine":   required=(activity, cadence),                 optional=(time_of_day, horizon)
  "vague_goal":required=(goal, first_action, weekly_cadence),optional=(horizon,)
  "lifestyle": required=(domains, cadence_per_domain, horizon), optional=(...)
}

def missing_required(plan_kind, filled_keys) -> list[str]:
    # 미충족 필수 슬롯 key 를 priority 순으로. 모르는 plan_kind 는 [] (추가 질문 없음)
```

예) `"매주 3번 헬스"` → routine, slots={activity:"헬스", cadence:"주3"} → `missing_required` = `[]`
→ **충분** → 바로 플랜. `"운동 하고 싶어"` → routine, slots={activity:"운동"} → `["cadence"]`
→ **부족** → 꼬리질문.


### 5.4 `follow_up` (`nodes/follow_up.py`) — 되묻기 + interrupt

- **무엇을:** 부족한 슬롯으로 자연스러운 한국어 질문 1개 생성 → `interrupt(question)` 로 멈춤.
  재개되면 `[assistant 질문, user 답변]` 두 줄을 `history` 에 append 하고 다시 `planner` 로.

```python
question = await ports.llm.generate_follow_up_question(missing_aspects=..., history=...)
user_answer = interrupt(question)          # ← 여기서 그래프가 멈추고 질문만 반환됨
return {"history": history + [{"assistant": question}, {"user": user_answer}], ...}
```

**왜 interrupt?** 멀티턴은 "질문 → (사용자 외부 응답) → 재개" 가 필요하다. LangGraph 의
`interrupt` + `MemorySaver` 체크포인트가 thread 별로 멈춘 지점을 저장해 다음 호출에 잇는다.

```mermaid
sequenceDiagram
    participant U as 사용자
    participant R as pipeline.run
    participant G as graph
    U->>R: "운동 하고 싶어"
    R->>G: 신규 실행
    G->>G: validate → planner(부족) → follow_up
    G-->>R: interrupt("주 몇 번 할까요?")
    R-->>U: FollowUpResult(question, thread_id)
    U->>R: "주 3번" + thread_id
    R->>G: Command(resume="주 3번")
    G->>G: follow_up 재개 → planner(충분) → plan_generator
    G-->>R: 최종 state(todos, events)
    R-->>U: GenerateResult
```

> **꼬리질문 ≤2회**(D9): `planner_node` 의 `_follow_up_count(...) >= 2` 가드. 그 이상이면
> 기본값을 채워 플랜을 낸다(사용자를 무한히 붙잡지 않는다).

> ⚠️ **LangGraph 함정:** `interrupt` 가 든 노드는 **resume 시 노드 처음부터 다시 실행**된다.
> 즉 `generate_follow_up_question` 은 (질문할 때 1번 + 재개될 때 1번) **두 번 불릴 수 있다.**
> 그래서 이 호출은 부수효과 없이(idempotent) 안전해야 한다. 아래 8절의 가짜 LLM 예제도
> 이 때문에 질문 함수는 큐를 소진하지 않고 **상수**를 돌려준다.


### 5.5 `plan_generator` (`nodes/plan_generator.py`) — 플랜 생성 + 날짜 정리

> ⚠️ **이 노드 내부가 Phase 2A 에서 "0단계 용량 → LLM 순서 → 코드 매핑 → critic" 분업으로
> 바뀔 자리다.** 아래는 **현재(Phase 0/1)** 동작.

```python
goal_tag = await llm.generate_goal_tag(...)                 # 목표 대표 태그 1개
summary, plan = await llm.generate_plan(parsed_goal, today) # LLM: 절대날짜 days[]
plan = _prepare_plan_days(plan, ...)    # 날짜 정리(같은 날만 있으면 today 부터 하루씩 펼침)
plan = _clamp_to_deadline(plan, deadline=...)   # [P1] 마감 이후 날짜 제거
# today==due_date → todos, 그 외 → calendar_events 로 분리
```

```mermaid
flowchart LR
    A["generate_plan<br/>(LLM, 절대날짜 days)"] --> B["_prepare_plan_days<br/>날짜 하루씩 정리·goal_tag 부착"]
    B --> C["_clamp_to_deadline<br/>마감 이후 제거 (P1)"]
    C --> D{"due_date == today?"}
    D -->|예| TODO["todos"]
    D -->|아니오| EV["calendar_events"]
```

> **P1 버그(Phase 0 핫픽스):** "일주일 뒤 시험" 이 6일차 시험·7일차 회고로 밀리던 문제 →
> `_clamp_to_deadline` 가 마감 이후를 잘라 보장. Phase 2A 는 여기에 **마감 앵커 매핑 +
> 용량 검사 critic** 을 더해 일반화한다.


### 5.6 `out_of_scope` (`nodes/out_of_scope.py`)

플랜과 무관한 입력(날씨·잡담)에는 고정 안내문만 반환한다.

```python
return {"out_of_scope_message": OUT_OF_SCOPE_MESSAGE, "todos": [], "calendar_events": []}
```


## 6. 보조 모듈 (결정적·LLM 무관)

### 6.1 `allocator.py` [Phase 1] — routine 전개

```python
expand_routine("헬스", "주 3회", today=..., horizon_days=28)
# → "주 N회"/"월수금" 을 horizon 내 실제 날짜로 펼쳐 TaskCandidate 리스트로
```

### 6.2 `date_parser.py` — 한국어 날짜 해석

```python
parse_explicit_deadline("3일 뒤 발표", today=...)      # → today+3
parse_explicit_deadline("다음주 화요일", today=...)     # → 해당 날짜
```

### 6.3 `_prompts.py` — 모든 프롬프트가 여기 모여 있다

`PLANNER_JUDGE_SYSTEM`(분류·충족), `FOLLOW_UP_SYSTEM`(질문), `PLAN_GENERATOR_SYSTEM`(플랜),
`GOAL_TAG_SYSTEM`(태그). 모델 출력은 **항상 JSON 객체 하나** 로 강제된다.


## 7. 끝까지 따라가보기 — 3가지 시나리오

### A. routine ("매주 3번 헬스") — 질문 없이 바로 플랜

```mermaid
flowchart LR
    I["매주 3번 헬스"] --> J["judge: routine<br/>slots{activity,cadence} 충족"]
    J --> P["sufficient=True"] --> G["plan_generator"] --> R["GenerateResult"]
```

### B. exam ("일주일 후 정처기 시험") — 마감 앵커

```mermaid
flowchart LR
    I["일주일 후 정처기 시험"] --> J["judge: exam, deadline=오늘+7"]
    J --> P["sufficient (마감 명확)"] --> G["plan_generator<br/>마감일=마지막날, 이후 clamp"] --> R["GenerateResult"]
```

### C. vague_goal ("운동 꾸준히 하고 싶어") — 1번 되묻고 플랜

```mermaid
flowchart LR
    I["운동 꾸준히"] --> J1["judge: vague_goal<br/>missing=[first_action...]"]
    J1 --> F["follow_up: '지금 가장 걸리는 건?'"]
    F -->|"사용자 답변"| J2["judge 재평가: 충족"]
    J2 --> G["plan_generator"] --> R["GenerateResult"]
```


## 8. 응답 형태 (`agents/todo_creation/schemas.py`)

`run()` 은 셋 중 하나를 돌려준다(모두 `{"result": ...}` 봉투):

| kind           | 언제                  | 핵심 필드                                      |
| -------------- | --------------------- | ---------------------------------------------- |
| `follow_up`    | 정보 부족 → interrupt | `question`, `missing_aspects`, `thread_id`     |
| `candidates`   | 플랜 완성             | `todos[]`, `calendar_events[]`, `summary_text` |
| `out_of_scope` | 범위 밖               | `message`                                      |

`thread_id` 를 다음 호출에 그대로 넣어야 멀티턴이 이어진다.


In [ ]:
# ── 모델 없이 한 번 돌려보기 (가짜 LLM) ──
# 진짜 RunPod 없이도 흐름을 눈으로 보도록, 고정 응답 LLM 으로 멀티턴을 재현한다.
from dataclasses import dataclass, field
from datetime import date, datetime

from agents.todo_creation.planner.pipeline import PlannerPorts, run
from agents.todo_creation.schemas import PlannerInput


@dataclass
class FakeLLM:
    sufficiency: list = field(
        default_factory=list
    )  # (sufficient, missing, parsed_goal) 큐
    # interrupt 노드는 resume 시 재실행되어 질문 함수가 두 번 불릴 수 있으므로 상수로 둔다(5.4 함정).
    follow_up: str = "좋아! 지금 가장 걸리는 건 뭐야?"

    async def judge_sufficiency(
        self, *, history, message, today, user_profile_memory=None
    ):
        return self.sufficiency.pop(0)

    async def generate_follow_up_question(self, *, missing_aspects, history):
        return self.follow_up

    async def generate_goal_tag(self, *, parsed_goal, history):
        return str(parsed_goal.get("goal_tag") or "목표")

    async def generate_plan(self, *, parsed_goal, today):
        return "요약: 차근차근 해봐요", [{"date": today, "tasks": []}]

    async def tag_plan(self, *, plan, parsed_goal):
        return plan

    async def split_tasks(self, *, prompt, today):  # 단일턴용(여기선 미사용)
        ...


today = date(2026, 6, 15)
now = datetime(2026, 6, 15, 9, 0)

# 시나리오 C: 한 번 되묻고(부족) → 답변 후 충족
llm = FakeLLM(
    sufficiency=[
        (
            False,
            ["first_action"],
            {
                "intent": "plan",
                "plan_kind": "vague_goal",
                "slots": {"goal": "운동"},
                "goal_text": "운동",
                "deadline": None,
            },
        ),
        (
            True,
            [],
            {
                "intent": "plan",
                "plan_kind": "vague_goal",
                "slots": {
                    "goal": "운동",
                    "first_action": "산책",
                    "weekly_cadence": "주3",
                },
                "goal_text": "운동",
                "goal_tag": "운동",
                "deadline": None,
            },
        ),
    ],
)
ports = PlannerPorts(llm=llm)

turn1 = await run(
    PlannerInput(user_id="u1", message="운동 꾸준히 하고 싶어", today=today),
    ports=ports,
    now=now,
)
print("turn1 →", turn1.kind, "|", getattr(turn1, "question", ""))

turn2 = await run(
    PlannerInput(
        user_id="u1", message="시간이 없어서", today=today, thread_id=turn1.thread_id
    ),
    ports=ports,
    now=now,
)
print(
    "turn2 →",
    turn2.kind,
    "| todos:",
    len(getattr(turn2, "todos", [])),
    "events:",
    len(getattr(turn2, "calendar_events", [])),
)

## 9. 다음 단계 — Phase 2A 예고 (어디가 바뀌나)

지금은 `plan_generator` 가 **모델에게 절대 날짜까지** 맡긴다(작은 모델엔 부담). Phase 2A 는
이 노드 **내부만** 뉴로-심볼릭 분업으로 교체한다(그래프 토폴로지·응답 형태는 그대로):

```mermaid
flowchart LR
    N0["0. 코드: 용량 N 선계산<br/>(남은 일수 × 하루 가용)"] --> N1["1. LLM: N개 task + 순서<br/>(절대 날짜 X)"]
    N1 --> N2["2. 코드: 마감 앵커 매핑 + clamp"]
    N2 --> N3["3. critic: 정합성 검증<br/>(우겨넣기 차단)"]
    N3 -->|"통과"| OUT["GenerateResult"]
    N3 -->|"용량 초과"| FIX["유연=축소 / 고정=통지"]
```

상세 계획: `docs/superpowers/plans/2026-06-15-daily-life-planner-phase2a.md`

---

_이 노트북은 `scripts/_gen_planner_notebook.py` 로 생성됨. 코드가 바뀌면 재생성하거나 직접 갱신할 것._
